In [ ]:
%pip install scikit-learn pandas

In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    cohen_kappa_score,
    confusion_matrix
)

In [ ]:
"""
Inter-Annotator Reliability Analysis
Cohen's Kappa

Input CSV format:
    Label_prediction,annotator1,annotator2

Example:
    "Microdata Request",TRUE,TRUE
    "Microdata Request",FALSE,FALSE
    "Microdata Request",TRUE,FALSE

Author: Yohanes Wahyu Trio Pramono
Python: 3.11.9
"""

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score


# ============================================================
# CONFIGURATION
# ============================================================

# Change this path to your CSV file.
INPUT_FILE = Path("inter-rater-reliability.csv")

# Output files
OUTPUT_SUMMARY = Path("iaa_summary.csv")
OUTPUT_CONTINGENCY = Path("iaa_contingency_table.csv")


# ============================================================
# FUNCTION: CLEAN BOOLEAN ANNOTATIONS
# ============================================================

def clean_boolean(value):
    """
    Convert different TRUE/FALSE representations
    into Python boolean values.

    Accepted examples:
        TRUE
        True
        true
        FALSE
        False
        false

    Returns:
        True
        False
        np.nan
    """

    if pd.isna(value):
        return np.nan

    value = str(value).strip().upper()

    if value == "TRUE":
        return True

    if value == "FALSE":
        return False

    return np.nan


# ============================================================
# FUNCTION: INTERPRET COHEN'S KAPPA
# ============================================================

def interpret_kappa(kappa):
    """
    Interpret Cohen's Kappa using the commonly used
    Landis and Koch scale.

    Note:
    This interpretation is a guideline rather than
    an absolute statistical standard.
    """

    if np.isnan(kappa):
        return "Undefined"

    if kappa < 0:
        return "Less than chance agreement"

    if kappa < 0.20:
        return "Slight agreement"

    if kappa < 0.40:
        return "Fair agreement"

    if kappa < 0.60:
        return "Moderate agreement"

    if kappa < 0.80:
        return "Substantial agreement"

    return "Almost perfect agreement"


# ============================================================
# 1. CHECK INPUT FILE
# ============================================================

print("=" * 70)
print("INTER-ANNOTATOR RELIABILITY ANALYSIS")
print("Cohen's Kappa")
print("=" * 70)

print(f"\nInput file:")
print(INPUT_FILE.resolve())

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"\nERROR: Input file not found:\n"
        f"{INPUT_FILE.resolve()}\n\n"
        f"Please change INPUT_FILE in the configuration section."
    )


# ============================================================
# 2. READ CSV
# ============================================================

df = pd.read_csv(INPUT_FILE)

print("\nDataset loaded successfully.")

print(f"Number of rows    : {len(df):,}")
print(f"Number of columns : {len(df.columns)}")

print("\nColumns:")
for column in df.columns:
    print(f"  - {column}")


# ============================================================
# 3. VALIDATE REQUIRED COLUMNS
# ============================================================

required_columns = [
    "Label_prediction",
    "annotator1",
    "annotator2"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        "\nERROR: Missing required columns:\n"
        + "\n".join(
            f"  - {column}"
            for column in missing_columns
        )
    )


# ============================================================
# 4. SELECT RELEVANT COLUMNS
# ============================================================

data = df[
    [
        "Label_prediction",
        "annotator1",
        "annotator2"
    ]
].copy()


# ============================================================
# 5. CLEAN ANNOTATOR VALUES
# ============================================================

data["annotator1_clean"] = (
    data["annotator1"]
    .apply(clean_boolean)
)

data["annotator2_clean"] = (
    data["annotator2"]
    .apply(clean_boolean)
)


# ============================================================
# 6. CHECK INVALID VALUES
# ============================================================

invalid_a1 = (
    data["annotator1"].notna()
    &
    data["annotator1_clean"].isna()
)

invalid_a2 = (
    data["annotator2"].notna()
    &
    data["annotator2_clean"].isna()
)

invalid_count = (
    invalid_a1.sum()
    +
    invalid_a2.sum()
)

if invalid_count > 0:

    print("\nWARNING:")
    print(
        f"{invalid_count} invalid annotation values "
        "were detected."
    )

    print("\nInvalid Annotator 1 values:")
    print(
        data.loc[
            invalid_a1,
            "annotator1"
        ].unique()
    )

    print("\nInvalid Annotator 2 values:")
    print(
        data.loc[
            invalid_a2,
            "annotator2"
        ].unique()
    )


# ============================================================
# 7. SELECT OVERLAPPING RECORDS
# ============================================================

original_count = len(data)

data_iaa = data.dropna(
    subset=[
        "annotator1_clean",
        "annotator2_clean"
    ]
).copy()

iaa_count = len(data_iaa)

excluded_count = (
    original_count - iaa_count
)

print("\n" + "=" * 70)
print("DATA USED FOR IAA")
print("=" * 70)

print(
    f"Original records       : {original_count:,}"
)

print(
    f"Records used for IAA   : {iaa_count:,}"
)

print(
    f"Records excluded       : {excluded_count:,}"
)


# ============================================================
# 8. CREATE ANNOTATOR LABELS
# ============================================================

annotator1 = data_iaa[
    "annotator1_clean"
]

annotator2 = data_iaa[
    "annotator2_clean"
]


# ============================================================
# 9. CONTINGENCY TABLE
# ============================================================

contingency = pd.crosstab(
    annotator1,
    annotator2,
    rownames=["Annotator 1"],
    colnames=["Annotator 2"]
)

# Ensure TRUE/FALSE always appear
contingency = contingency.reindex(
    index=[True, False],
    columns=[True, False],
    fill_value=0
)

print("\n" + "=" * 70)
print("CONTINGENCY TABLE")
print("=" * 70)

print(
    "\n                    Annotator 2"
)

print(
    "                    TRUE    FALSE"
)

print(
    f"Annotator 1 TRUE    "
    f"{contingency.loc[True, True]:>5}    "
    f"{contingency.loc[True, False]:>5}"
)

print(
    f"Annotator 1 FALSE   "
    f"{contingency.loc[False, True]:>5}    "
    f"{contingency.loc[False, False]:>5}"
)


# ============================================================
# 10. EXTRACT AGREEMENT COUNTS
# ============================================================

true_true = int(
    contingency.loc[True, True]
)

true_false = int(
    contingency.loc[True, False]
)

false_true = int(
    contingency.loc[False, True]
)

false_false = int(
    contingency.loc[False, False]
)

N = (
    true_true
    +
    true_false
    +
    false_true
    +
    false_false
)


# ============================================================
# 11. OBSERVED AGREEMENT
# ============================================================

observed_agreement = (
    true_true + false_false
) / N


# ============================================================
# 12. EXPECTED AGREEMENT
# ============================================================

# Annotator 1 marginal proportions

a1_true = (
    true_true + true_false
) / N

a1_false = (
    false_true + false_false
) / N


# Annotator 2 marginal proportions

a2_true = (
    true_true + false_true
) / N

a2_false = (
    true_false + false_false
) / N


expected_agreement = (
    a1_true * a2_true
    +
    a1_false * a2_false
)


# ============================================================
# 13. COHEN'S KAPPA
# ============================================================

if expected_agreement == 1:

    kappa_manual = np.nan

else:

    kappa_manual = (
        observed_agreement
        -
        expected_agreement
    ) / (
        1
        -
        expected_agreement
    )


# ============================================================
# 14. VERIFY USING SCIKIT-LEARN
# ============================================================

kappa_sklearn = cohen_kappa_score(
    annotator1,
    annotator2
)


# ============================================================
# 15. CHECK CALCULATION
# ============================================================

if np.isclose(
    kappa_manual,
    kappa_sklearn,
    equal_nan=True
):

    calculation_status = "PASS"

else:

    calculation_status = "WARNING"


# ============================================================
# 16. INTERPRETATION
# ============================================================

interpretation = interpret_kappa(
    kappa_sklearn
)


# ============================================================
# 17. PRINT RESULTS
# ============================================================

print("\n" + "=" * 70)
print("AGREEMENT STATISTICS")
print("=" * 70)

print(
    f"\nTotal overlapping records : {N:,}"
)

print(
    f"TRUE / TRUE              : {true_true:,}"
)

print(
    f"TRUE / FALSE             : {true_false:,}"
)

print(
    f"FALSE / TRUE             : {false_true:,}"
)

print(
    f"FALSE / FALSE            : {false_false:,}"
)

print(
    f"\nObserved agreement       : "
    f"{observed_agreement:.6f}"
)

print(
    f"Observed agreement       : "
    f"{observed_agreement * 100:.2f}%"
)

print(
    f"\nExpected agreement       : "
    f"{expected_agreement:.6f}"
)

print(
    f"Expected agreement       : "
    f"{expected_agreement * 100:.2f}%"
)

print(
    f"\nCohen's Kappa (manual)    : "
    f"{kappa_manual:.6f}"
)

print(
    f"Cohen's Kappa (sklearn)  : "
    f"{kappa_sklearn:.6f}"
)

print(
    f"\nInterpretation            : "
    f"{interpretation}"
)

print(
    f"Calculation verification : "
    f"{calculation_status}"
)


# ============================================================
# 18. ANNOTATOR DISTRIBUTION
# ============================================================

a1_distribution = (
    annotator1
    .value_counts()
    .reindex([True, False], fill_value=0)
)

a2_distribution = (
    annotator2
    .value_counts()
    .reindex([True, False], fill_value=0)
)

print("\n" + "=" * 70)
print("ANNOTATOR DISTRIBUTION")
print("=" * 70)

print("\nAnnotator 1:")

print(
    f"  TRUE  : {a1_distribution[True]:,} "
    f"({a1_distribution[True] / N * 100:.2f}%)"
)

print(
    f"  FALSE : {a1_distribution[False]:,} "
    f"({a1_distribution[False] / N * 100:.2f}%)"
)

print("\nAnnotator 2:")

print(
    f"  TRUE  : {a2_distribution[True]:,} "
    f"({a2_distribution[True] / N * 100:.2f}%)"
)

print(
    f"  FALSE : {a2_distribution[False]:,} "
    f"({a2_distribution[False] / N * 100:.2f}%)"
)


# ============================================================
# 19. PREVALENCE / IMBALANCE CHECK
# ============================================================

true_rate_a1 = a1_distribution[True] / N
true_rate_a2 = a2_distribution[True] / N

print("\n" + "=" * 70)
print("PREVALENCE CHECK")
print("=" * 70)

print(
    f"\nAnnotator 1 TRUE rate : "
    f"{true_rate_a1 * 100:.2f}%"
)

print(
    f"Annotator 2 TRUE rate : "
    f"{true_rate_a2 * 100:.2f}%"
)

if (
    true_rate_a1 > 0.90
    or
    true_rate_a2 > 0.90
):

    print(
        "\nWARNING: TRUE is highly prevalent "
        "in the annotation data."
    )

    print(
        "Cohen's Kappa may be affected by "
        "the prevalence problem."
    )

else:

    print(
        "\nTRUE/FALSE distribution does not show "
        "extreme prevalence based on the 90% threshold."
    )


# ============================================================
# 20. SAVE RESULTS
# ============================================================

# Create output directory if necessary
OUTPUT_SUMMARY.parent.mkdir(
    parents=True,
    exist_ok=True
)


# Summary table
summary = pd.DataFrame({
    "Measure": [
        "Total overlapping records",
        "TRUE / TRUE",
        "TRUE / FALSE",
        "FALSE / TRUE",
        "FALSE / FALSE",
        "Observed agreement",
        "Expected agreement",
        "Cohen's Kappa",
        "Interpretation"
    ],
    
    "Value": [
        N,
        true_true,
        true_false,
        false_true,
        false_false,
        observed_agreement,
        expected_agreement,
        kappa_sklearn,
        interpretation
    ]
})


summary.to_csv(
    OUTPUT_SUMMARY,
    index=False
)


# Contingency table
contingency.to_csv(
    OUTPUT_CONTINGENCY
)


# ============================================================
# 21. FINAL MESSAGE
# ============================================================

print("\n" + "=" * 70)
print("OUTPUT FILES")
print("=" * 70)

print(
    f"\nSummary:"
    f"\n  {OUTPUT_SUMMARY.resolve()}"
)

print(
    f"\nContingency table:"
    f"\n  {OUTPUT_CONTINGENCY.resolve()}"
)

print("\nAnalysis completed successfully.")